# Neural Style Transfer — A Neural Algorithm of Artistic Style (2015)

_Papers · Generative Models_

**Keep one photo's CONTENT and another painting's STYLE by optimizing a new image's pixels against two feature-space losses read off a frozen pretrained CNN.**

---

This notebook is a practice scaffold for the **Neural Style Transfer — A Neural Algorithm of Artistic Style (2015)** lesson. We build it one piece at a time: a frozen VGG feature extractor, a content loss and a Gram-matrix style loss, then optimizing the image's pixels against both. Run each cell, read the note above it, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np
import matplotlib.pyplot as plt

## Reference implementation — PyTorch

### Step 1 — Sanity-check the Gram matrix on a tiny example

Style in this paper is captured by the **Gram matrix** `G = F Fᵀ` (Eq. 3), which measures how strongly each pair of feature channels co-activates — and crucially ignores *where* in the image they fire. We verify two things on a 2-channel toy: the Gram formula reproduces `[[5,2],[2,2]]`, and shuffling the spatial positions leaves the Gram matrix unchanged, confirming it is position-free. We also compute one per-layer style error `E_l` (Eq. 4).

In [ ]:
import torch, torch.nn as nn, torch.nn.functional as Fn
import torchvision

torch.manual_seed(0)
device = "cuda" if torch.cuda.is_available() else "cpu"

# --- 0. Sanity-check the worked example: the Gram matrix (Eq. 3) and a style error (Eq. 4). ---
F0 = torch.tensor([[1., 2., 0.],
                   [0., 1., 1.]])              # N=2 channels, M=3 positions
G0 = F0 @ F0.t()                               # Eq. 3:  G_ij = sum_k F_ik F_jk
print("worked example -- Gram matrix G = F F^T:")
print(G0)                                      # [[5,2],[2,2]]

F0s = F0[:, [2, 0, 1]]                          # shuffle the 3 positions
print("position-shuffled gives the SAME Gram (style is position-free):")
print(F0s @ F0s.t())                           # still [[5,2],[2,2]]

A0 = torch.tensor([[5., 3.], [3., 2.]])         # a toy style target
N, M = 2, 3
El = ((G0 - A0) ** 2).sum() / (4 * N**2 * M**2) # Eq. 4
print("per-layer style loss E_l = %.4f  (= 2/144)" % El.item())   # 0.0139

### Step 2 — Load a frozen VGG-19 and helper functions

The network is a fixed lens, not something we train: we load pretrained VGG-19, freeze every weight, and (per the paper) swap max-pooling for average-pooling so the image gradients are smoother. We pick which layers read style (conv1_1..conv5_1) versus content (conv4_2), then define `features` (collect those layers' maps after ImageNet normalization) and `gram` (the raw `F Fᵀ`).

In [ ]:
# --- 1. A FROZEN pretrained VGG-19 feature extractor; we read specific layers' maps. ---
weights = torchvision.models.VGG19_Weights.DEFAULT
vgg = torchvision.models.vgg19(weights=weights).features.to(device).eval()
for p in vgg.parameters():                      # FREEZE every weight: the net never changes.
    p.requires_grad_(False)

# Paper: average pooling gives smoother image gradients than max pooling.
for i, m in enumerate(vgg):
    if isinstance(m, nn.MaxPool2d):
        vgg[i] = nn.AvgPool2d(kernel_size=2, stride=2)

# Indices into vgg.features for conv1_1..conv5_1 (style) and conv4_2 (content) in VGG-19.
STYLE_LAYERS   = {0: "conv1_1", 5: "conv2_1", 10: "conv3_1", 19: "conv4_1", 28: "conv5_1"}
CONTENT_LAYER  = 21                              # conv4_2
WL = 1.0 / len(STYLE_LAYERS)                     # w_l = 1/5 (Eq. 5)

# ImageNet normalization (pretrained VGG expects it).
MEAN = torch.tensor([0.485, 0.456, 0.406], device=device).view(1, 3, 1, 1)
STD  = torch.tensor([0.229, 0.224, 0.225], device=device).view(1, 3, 1, 1)

def norm(x):
    return (x - MEAN) / STD

def features(x):                                # run x through VGG, collect the layers we need
    feats = {}
    x = norm(x)
    for i, layer in enumerate(vgg):
        x = layer(x)
        if i == CONTENT_LAYER or i in STYLE_LAYERS:
            feats[i] = x
    return feats

def gram(f):                                     # Eq. 3: F F^T over the spatial dimension
    b, c, h, w = f.shape                          # N = c channels, M = h*w positions
    F = f.view(c, h * w)
    return (F @ F.t()) / 1.0                       # raw Gram; the 1/(4 N^2 M^2) lives in Eq. 4

### Step 3 — Make the images and precompute the fixed targets

To keep the notebook download-free we synthesize two small images: a content image (a soft disk — a clear layout to preserve) and a style image (diagonal colour stripes — a position-free texture). We then run each through VGG **once** to freeze the targets: the content feature map `P` and the style Gram matrices `A`. These are detached so gradients later flow only into the canvas, not these references.

In [ ]:
# --- 2. Two SMALL images: a content image and a style image (toy patterns, no downloads). ---
def make_imgs(size=64):
    yy, xx = torch.meshgrid(torch.linspace(0, 1, size), torch.linspace(0, 1, size), indexing="ij")
    # content: a soft disk on a plain field (clear LAYOUT to preserve)
    disk = ((xx - 0.5)**2 + (yy - 0.5)**2 < 0.08).float()
    content = torch.stack([0.2 + 0.6*disk, 0.3 + 0.3*disk, 0.6 - 0.3*disk])
    # style: diagonal colour stripes (a TEXTURE, position-free statistics)
    stripes = (((xx + yy) * 6) % 1.0 < 0.5).float()
    style = torch.stack([0.9*stripes, 0.2 + 0.5*stripes, 0.9 - 0.7*stripes])
    return content.unsqueeze(0).to(device), style.unsqueeze(0).to(device)

content_img, style_img = make_imgs()

# Precompute fixed targets P^l (content) and A^l (style Gram), DETACHED (Eqs. 1, 3).
with torch.no_grad():
    P = features(content_img)[CONTENT_LAYER].detach()
    A = {i: gram(features(style_img)[i]).detach() for i in STYLE_LAYERS}

### Step 4 — Define the two losses, then optimize the pixels

Now the heart of the method. The content loss (Eq. 1) is the squared distance between the canvas's deep features and `P`; the style loss (Eqs. 4–5) sums the squared Gram-matrix distances to `A` across layers. The optimizer's variable is the **image itself** (not the network), driven by L-BFGS to minimize `α·content + β·style` (Eq. 7). We run it once with style on and once as a content-only ablation (β = 0); turning style on should drive the style loss far lower.

In [ ]:
# --- 3. The losses (Eq. 1, Eq. 4/5) and the TOTAL (Eq. 7). ---
def content_loss(feats):                          # Eq. 1: 1/2 * sum (F - P)^2
    return 0.5 * ((feats[CONTENT_LAYER] - P) ** 2).sum()

def style_loss(feats):                             # Eqs. 4 + 5
    total = 0.0
    for i in STYLE_LAYERS:
        f = feats[i]
        _, c, h, w = f.shape
        El = ((gram(f) - A[i]) ** 2).sum() / (4 * (c ** 2) * ((h * w) ** 2))
        total = total + WL * El
    return total

# --- 4. Optimize the IMAGE PIXELS (not the weights) with L-BFGS. ---
def stylize(alpha, beta, steps=20):
    img = content_img.clone().requires_grad_(True)   # THE IMAGE is the only variable
    opt = torch.optim.LBFGS([img], max_iter=steps, lr=0.5)
    hist = {}
    def closure():
        opt.zero_grad()
        feats = features(img.clamp(0, 1))
        c = content_loss(feats)
        s = style_loss(feats)
        loss = alpha * c + beta * s                  # Eq. 7
        loss.backward()
        hist["content"], hist["style"] = c.item(), s.item()
        return loss
    opt.step(closure)
    return img.detach().clamp(0, 1), hist

print("\n--- content+style (Eq. 7, alpha/beta = 1e-3) ---")
out_both, h_both = stylize(alpha=1.0, beta=1e3, steps=20)
print("content loss %.3e   style loss %.3e" % (h_both["content"], h_both["style"]))

print("\n--- ABLATION: content-only (beta = 0) ---")
out_content, h_c = stylize(alpha=1.0, beta=0.0, steps=20)
print("content loss %.3e   style loss %.3e (style not optimized)" % (h_c["content"], h_c["style"]))

print("\nWith style ON, the style loss is far lower than content-only -> the Gram term paints the texture in.")
print("(Our small run, not the paper's reported numbers.)")

## Visualize the data & results

_Does the Gram-matrix style loss (Eq. 4/5) actually paint the style in — and what does the content-only ablation (β = 0) leave out?_

### Step 1 — Rebuild the frozen extractor and toy images

The visualization is self-contained, so it re-creates the pieces it needs: a frozen, average-pooled VGG-19, the style/content layer choices, the ImageNet normalization, and the `features`/`gram` helpers. It then regenerates the same disk content image and striped style image and freezes their targets `P` and `A`, exactly as in the reference run above.

In [ ]:
import torch, torch.nn as nn, torchvision
torch.manual_seed(0)
device = "cpu"

# Frozen VGG-19 features; average pooling per the paper.
vgg = torchvision.models.vgg19(weights=torchvision.models.VGG19_Weights.DEFAULT).features.eval()
for p in vgg.parameters():
    p.requires_grad_(False)
for i, m in enumerate(vgg):
    if isinstance(m, nn.MaxPool2d):
        vgg[i] = nn.AvgPool2d(2, 2)

STYLE = {0, 5, 10, 19, 28}
CONTENT = 21
WL = 1 / len(STYLE)
MEAN = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1)
STD  = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1)

def feats(x):
    out = {}
    x = (x - MEAN) / STD
    for i, L in enumerate(vgg):
        x = L(x)
        if i == CONTENT or i in STYLE:
            out[i] = x
    return out

def gram(f):
    _, c, h, w = f.shape
    F = f.view(c, h * w)
    return F @ F.t()

def imgs(s=64):
    yy, xx = torch.meshgrid(torch.linspace(0, 1, s), torch.linspace(0, 1, s), indexing="ij")
    disk = ((xx - .5)**2 + (yy - .5)**2 < .08).float()
    content = torch.stack([.2 + .6*disk, .3 + .3*disk, .6 - .3*disk]).unsqueeze(0)
    stripes = (((xx + yy) * 6) % 1. < .5).float()
    style = torch.stack([.9*stripes, .2 + .5*stripes, .9 - .7*stripes]).unsqueeze(0)
    return content, style

content_img, style_img = imgs()
with torch.no_grad():
    P = feats(content_img)[CONTENT].detach()
    A = {i: gram(feats(style_img)[i]).detach() for i in STYLE}

### Step 2 — Run the content-vs-style ablation and compare

With the losses and optimizer in place, we run the pixel optimization twice: once with both terms on (`beta = 1e3`) and once content-only (`beta = 0`). The printed ratio of style losses (both / content-only) is the key read-out — it should be well below 1, the numeric signature that switching the Gram term on is what paints the style in while content stays low.

In [ ]:
def closs(F):
    return 0.5 * ((F[CONTENT] - P) ** 2).sum()

def sloss(F):
    t = 0.
    for i in STYLE:
        f = F[i]
        _, c, h, w = f.shape
        t = t + WL * ((gram(f) - A[i]) ** 2).sum() / (4 * c**2 * (h * w)**2)
    return t

def run(alpha, beta, steps=20):
    img = content_img.clone().requires_grad_(True)
    opt = torch.optim.LBFGS([img], max_iter=steps, lr=0.5)
    log = {}
    def cl():
        opt.zero_grad()
        F = feats(img.clamp(0, 1))
        c, s = closs(F), sloss(F)
        loss = alpha * c + beta * s
        loss.backward()
        log['c'], log['s'] = c.item(), s.item()
        return loss
    opt.step(cl)
    return log

both = run(1.0, 1e3)
only = run(1.0, 0.0)
print("content+style : content %.3e  style %.3e" % (both['c'], both['s']))
print("content-only  : content %.3e  style %.3e" % (only['c'], only['s']))
print("style loss ratio (both / content-only) = %.3f" % (both['s'] / only['s']))
# Style ON drives the style loss far below the content-only value -> the Gram term paints style in.
# Our small run, not the paper's reported numbers.

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** The content-vs-style ablation. Run the optimizer with $\beta=0$ (content term only), then again
            with both terms on (same content layer, same start, same steps). What does the content-only image look
            like, and what exactly does turning on the Gram-matrix style loss add?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Set $\beta=0$ so the loss is purely $\alpha\mathcal{L}_{content}$ (Eq. 1) and optimize the pixels. — _An honest ablation changes one thing: whether the style (Gram) term is present._
- Observe: the result reconstructs the content image's deep features &mdash; it looks like the photo (its layout and objects), with none of the style image's colours or brush textures. — _Eq. (1) only asks the deep features to match the photo; nothing references the style image._
- Now switch on $\beta\,\mathcal{L}_{style}$ (Eqs. 3&ndash;5) and re-optimize. — _The Gram-matrix term injects the style image's channel correlations &mdash; its palette and textures &mdash; while content keeps the layout._
- Measure: the style loss against the painting drops sharply once $\beta\gt0$, while content stays low; the image now shows the photo's structure painted in the style's colours. — _The two losses are complementary &mdash; layout-preserving content vs position-free style._

**Answer:** With $\beta=0$ the optimized image is essentially a content reconstruction: it matches the
                 photo's deep features (its scene and layout) and shows none of the style image &mdash; no
                 borrowed colours, no brushwork. Turning on the Gram-matrix style term is exactly what paints the
                 style in: it drives the generated image's feature correlations toward the painting's, so the
                 style's palette and textures appear on top of the preserved content. In our tiny run the
                 style loss (distance of the canvas Gram matrices to the style image's) falls from its
                 content-only value to a far smaller one once $\beta\gt0$, while content stays low &mdash; the
                 numeric signature of "content kept, style added". (Our small run, not the paper's numbers.)

</details>

**Problem 2.** Your worked example had $F=\begin{bmatrix}1 & 2 & 0\\ 0 & 1 & 1\end{bmatrix}$ with Gram
            $G=\begin{bmatrix}5 & 2\\ 2 & 2\end{bmatrix}$. Shuffle the three spatial positions to
            $F'=\begin{bmatrix}2 & 0 & 1\\ 1 & 1 & 0\end{bmatrix}$. Compute $G'=F'F'^{\top}$ and say
            what it demonstrates.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- $G'_{11}=2^2+0^2+1^2=4+0+1=5$. — _Channel 1's energy is a sum over positions; reordering positions does not change a sum._
- $G'_{22}=1^2+1^2+0^2=1+1+0=2$. — _Same for channel 2._
- $G'_{12}=2\cdot1+0\cdot1+1\cdot0=2+0+0=2$. — _The cross term is a dot product over positions; the same pairs are multiplied, just in a different order._

**Answer:** $G'=\begin{bmatrix}5 & 2\\ 2 & 2\end{bmatrix}=G$ &mdash; identical to the original
                 Gram matrix even though the spatial arrangement changed. This is the key property: the Gram matrix
                 (Eq. 3) is invariant to where features fire and depends only on which channels co-activate.
                 That is precisely why it captures style/texture rather than content/layout, and why the
                 style loss can be satisfied without disturbing the content loss.

</details>

**Problem 3.** A classmate puts vgg.parameters() into the optimizer and watches the loss drop, but the
            input image never changes. What did they get backwards, and what is the correct setup?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Recall Eq. (7) is minimized with respect to the image $\vec{x}$, with the network held fixed. — _The paper's whole idea is to optimize in image space, not weight space._
- See the bug: optimizing vgg.parameters() changes the network, leaving the image untouched &mdash; the opposite of style transfer. — _Whatever tensor the optimizer holds is the thing that gets updated._
- Fix it: freeze the net (requires_grad_(False) on all weights, eval mode) and make the image a leaf with requires_grad_(True); pass [image] to the optimizer. — _Then autograd's gradient flows back to the pixels and the optimizer edits the canvas._

**Answer:** They optimized the wrong variable. Neural style transfer freezes the pretrained network and
                 treats the output image's pixels as the parameters (Eq. 7 is minimized over $\vec{x}$). Putting
                 vgg.parameters() in the optimizer trains the network instead, so the image stays
                 put. Correct setup: requires_grad_(False) on every VGG weight, the image tensor created
                 with requires_grad_(True), and only [image] handed to the optimizer.

</details>